# Kiểm tra hai nguồn dữ liệu trước khi gộp
Notebook chuẩn bị tích hợp dữ liệu của dự án Học máy. Giảng viên yêu cầu ít nhất hai bộ dữ liệu và gộp chúng; hiện chưa chốt nguồn B. Bạn tự chạy từng ô. Không huấn luyện mô hình và không sửa dữ liệu gốc.

**Chưa có nguồn B được xác nhận có thể gộp.** Cùng tên cột chưa đủ: cần cùng ý nghĩa, đơn vị, kỳ tổng hợp lượng mưa, nhãn và thông tin nguồn. Một bản sao của 2.200 dòng không làm tăng lượng thông tin. Dữ liệu sinh từ CTGAN không được coi là phép đo độc lập.

Đọc `HUONG_DAN_HAI_NGUON_DU_LIEU.md` trước khi chạy. Phần xuất file mặc định bị tắt; chỉ bật sau khi bạn đã có căn cứ về tính tương thích và nguồn gốc.

## 1. Chuẩn bị trên máy hoặc Colab
Local: mở notebook trong thư mục dự án. Colab: tải lên hai CSV bằng thanh Files, sau đó sửa hai đường dẫn ở ô dưới, ví dụ `/content/Crop_recommendation.csv` và `/content/nguon_b.csv`. Notebook chỉ dùng pandas và numpy có trong requirements của dự án.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

PATH_A = Path('data/Crop_recommendation.csv')
PATH_B = Path('data/nguon_b.csv')  # Bạn tải và chọn file nguồn B, không dùng file tự sinh.
FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
COLUMNS = FEATURES + ['label']

# Chỉ điền khi đã đối chiếu từ điển dữ liệu của nguồn B.
# Ví dụ: {'Nitrogen': 'N', 'Crop': 'label'}; không suy ra ý nghĩa chỉ từ tên gần giống.
RENAME_B = {}
LABEL_MAP_B = {}  # Chỉ ánh xạ các tên đã xác minh là cùng loại cây.

SOURCE_A = {'name': 'Atharva Ingle Crop Recommendation',
            'url': 'https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset',
            'license': 'Apache 2.0'}
SOURCE_B = {'name': '', 'url': '', 'license': '', 'download_date': ''}
SAME_MEANING_AND_UNITS = False  # Cần cả kỳ tích lũy rainfall, thang N/P/K và quy tắc gán nhãn.
INDEPENDENT_ORIGIN_CONFIRMED = False
SOURCE_B_IS_SYNTHETIC = None  # Điền True/False theo tài liệu nguồn; None là chưa biết.
EVIDENCE_NOTES = ''  # Ghi căn cứ xác minh; không chỉ đổi các cờ thành True để xuất file.
EXPORT_MERGED = False
OUT = Path('data/merged_candidate')


## 2. Đọc hai CSV và kiểm tra cấu trúc
Nếu thiếu cột, dừng để kiểm tra dữ liệu. Không tự điền các đặc trưng còn thiếu bằng 0 hoặc giá trị ngẫu nhiên. Việc bỏ các cột phụ trong ô dưới chỉ chọn đúng bảy đầu vào của đề tài; hãy lưu và đọc tài liệu của nguồn để giữ thông tin địa điểm/thời gian nếu có.

In [ ]:
raw_a = pd.read_csv(PATH_A)
raw_b = pd.read_csv(PATH_B)
display(pd.DataFrame({'nguon': ['A', 'B'], 'so_dong': [len(raw_a), len(raw_b)],
                      'cac_cot': [list(raw_a.columns), list(raw_b.columns)]}))

def prepare(frame, rename=None, label_map=None):
    result = frame.rename(columns=rename or {}).copy()
    if result.columns.duplicated().any():
        raise ValueError('Tên cột bị trùng sau ánh xạ; kiểm tra RENAME_B.')
    missing = set(COLUMNS) - set(result.columns)
    if missing:
        raise ValueError(f'Thiếu cột {missing}; chưa thể gộp theo cấu trúc hiện tại.')
    result = result[COLUMNS].copy()
    if result.isna().any().any():
        raise ValueError('Có giá trị thiếu; cần phân tích và xác định phương pháp xử lý trước.')
    result[FEATURES] = result[FEATURES].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(result[FEATURES].to_numpy(dtype=float)).all():
        raise ValueError('Có giá trị vô cực hoặc không hợp lệ.')
    result['label'] = result['label'].astype(str).str.strip().str.lower()
    result['label'] = result['label'].replace(label_map or {})
    if result['label'].eq('').any():
        raise ValueError('Có nhãn rỗng.')
    return result

a = prepare(raw_a)
b = prepare(raw_b, RENAME_B, LABEL_MAP_B)
display(pd.concat([a[FEATURES].describe().T.add_prefix('A_'),
                   b[FEATURES].describe().T.add_prefix('B_')], axis=1))
display(pd.concat([a['label'].value_counts().rename('A'),
                   b['label'].value_counts().rename('B')], axis=1).fillna(0))
print('Nhãn chỉ xuất hiện ở B:', sorted(set(b['label']) - set(a['label'])))
print('Nguồn gốc/đơn vị phải kiểm tra bằng tài liệu; thống kê tương tự không chứng minh tương thích.')


## 3. Kiểm tra trùng, xung đột nhãn và miền giá trị
Đếm trùng sau chuẩn hóa tên cột/nhãn. Ngoài trùng chính xác, phép làm tròn dưới đây chỉ cảnh báo bản sao có khác độ chính xác số; chưa dùng để tự xóa. Nếu hai nguồn giống nhau hoàn toàn, không gộp để ghi thành hai nguồn độc lập.

In [ ]:
a_unique = a.drop_duplicates(COLUMNS)
b_unique = b.drop_duplicates(COLUMNS)
overlap = b_unique.merge(a_unique, on=COLUMNS, how='inner')
new_b = b_unique.merge(a_unique.assign(_in_a=True), on=COLUMNS, how='left')
new_b = new_b.loc[new_b['_in_a'].isna(), COLUMNS]

joined = pd.concat([a.assign(source='A'), b.assign(source='B')], ignore_index=True)
conflict_counts = joined.groupby(FEATURES, dropna=False)['label'].nunique()
conflicting_features = conflict_counts[conflict_counts > 1]

rounded = joined.copy()
rounded[FEATURES] = rounded[FEATURES].round(6)
near_groups = rounded.groupby(COLUMNS)['source'].nunique()
print('Trùng nội bộ A:', len(a) - len(a_unique))
print('Trùng nội bộ B:', len(b) - len(b_unique))
print('B trùng chính xác với A:', len(overlap), '/', len(b_unique))
print('Dòng phân biệt B bổ sung:', len(new_b))
print('Nhóm cùng đặc trưng nhưng khác nhãn:', len(conflicting_features))
print('Nhóm trùng giữa nguồn sau làm tròn 6 chữ số:', int((near_groups > 1).sum()))

def invalid_domain(frame):
    return ((frame[['N', 'P', 'K', 'rainfall']] < 0).any(axis=1)
            | ~frame['humidity'].between(0, 100)
            | ~frame['ph'].between(0, 14))
print('Dòng ngoài miền đã kiểm tra:', int(invalid_domain(joined).sum()))
print('Hash file A:', hashlib.sha256(PATH_A.read_bytes()).hexdigest())
print('Hash file B:', hashlib.sha256(PATH_B.read_bytes()).hexdigest())


## 4. Xuất bản gộp khi đã xác minh
Giữ `EXPORT_MERGED = False` nếu nguồn B là dữ liệu sinh, chưa rõ đơn vị, có nhãn ngoài 22 lớp, hoặc chưa rõ quan hệ với nguồn A. Khi nguồn B là bản sao, kết luận “không bổ sung mẫu mới” là kết quả kiểm tra hợp lệ.

Ô này nối thêm dòng bằng `pd.concat`, không ghép ngang theo số thứ tự dòng. Nó giữ dấu vết nguồn trong file riêng; các cột nguồn không được dùng làm đặc trưng học. Các nhóm gần trùng phải được xem xét thêm khi thiết kế chia tập, kể cả khi không trùng chính xác.

In [ ]:
if not EXPORT_MERGED:
    print('Chưa xuất file. Hãy đọc kết quả kiểm tra và xác minh nguồn B trước.')
else:
    assert SAME_MEANING_AND_UNITS and INDEPENDENT_ORIGIN_CONFIRMED, 'Chưa xác minh tương thích/nguồn gốc.'
    assert SOURCE_B_IS_SYNTHETIC is False, 'Chưa dùng dữ liệu sinh trong quy trình gộp này.'
    assert EVIDENCE_NOTES.strip(), 'Cần ghi căn cứ xác minh.'
    assert all(SOURCE_B.values()), 'Thiếu hồ sơ nguồn B.'
    assert not (set(b['label']) - set(a['label'])), 'Có nhãn mới; cần thống nhất phạm vi đề tài trước.'
    assert len(conflicting_features) == 0, 'Cần xử lý xung đột nhãn trước.'
    assert not invalid_domain(joined).any(), 'Cần xem lại các dòng ngoài miền.'
    assert len(new_b) > 0, 'Nguồn B không bổ sung mẫu phân biệt.'
    assert not OUT.exists(), 'Thư mục đích đã có; chọn tên mới để giữ kết quả trước.'
    merged = joined.drop_duplicates(COLUMNS).reset_index(drop=True)
    lineage = joined.groupby(COLUMNS, sort=False)['source'].agg(lambda x: '+'.join(sorted(set(x))))
    lineage = lineage.reset_index(name='sources')
    lineage = merged[COLUMNS].merge(lineage, on=COLUMNS, how='left', validate='one_to_one')
    lineage.insert(0, 'merged_row', range(len(lineage)))
    manifest = {'source_a': SOURCE_A, 'source_b': SOURCE_B,
                'sha256_a': hashlib.sha256(PATH_A.read_bytes()).hexdigest(),
                'sha256_b': hashlib.sha256(PATH_B.read_bytes()).hexdigest(),
                'evidence': EVIDENCE_NOTES, 'rows_a': len(a), 'rows_b': len(b),
                'new_unique_rows_b': len(new_b), 'merged_rows': len(merged)}
    OUT.mkdir(parents=True)
    merged[COLUMNS].to_csv(OUT/'Crop_recommendation_merged.csv', index=False)
    lineage.to_csv(OUT/'row_sources.csv', index=False)
    (OUT/'sources.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf8')
    print('Đã xuất bản ứng viên:', OUT, '| số dòng:', len(merged))


## 5. Sau khi gộp
Chưa thay CSV của notebook chính. Đọc lại phân bố, kiểm tra trùng/gần trùng, nhãn và tính độc lập. Nếu dùng dữ liệu mới, tạo một lần thí nghiệm riêng, đổi `DATA_PATH` và `RUN_NAME`, giữ nguyên `outputs/lan_01`.

Chỉ chia train/test sau khi xử lý các nhóm trùng; không để cùng quan sát nằm ở cả hai phía. Nếu nguồn B là dữ liệu sinh từ nguồn A mà không biết tập dùng để sinh, không dùng B như tập kiểm tra độc lập. Không thêm tên nguồn vào `FEATURES`. Mọi thống kê và hình trong báo cáo phải cập nhật theo dữ liệu thực tế bạn đã dùng.